In [1]:
# Get HEP Data from GitHub Repo
# Unclear how to get zip file direct from HEP Data

import sys
import os

# 3. Download data
!wget https://raw.githubusercontent.com/BrownParticleAstro/dmtools_notebooks/refs/heads/main/recent_data/ins2841863/SIcrosssection.csv


--2026-10-02 09:30:07--  https://raw.githubusercontent.com/BrownParticleAstro/dmtools_notebooks/refs/heads/main/recent_data/ins2841863/SIcrosssection.csv
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.109.133, 185.199.108.133, 185.199.111.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.109.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 6246 (6.1K) [text/plain]
Saving to: ‘SIcrosssection.csv’

SIcrosssection.csv  100%[===================>]   6.10K  --.-KB/s    in 0s      

2026-10-02 09:30:08 (79.7 MB/s) - ‘SIcrosssection.csv’ saved [6246/6246]



In [2]:
import csv

# Read the downloaded CSV file into a list of lists
with open('SIcrosssection.csv', 'r') as file:
    reader = csv.reader(file)
    data_list = list(reader)

# Display the first few rows of the list of lists
print("First 5 rows of data (list of lists):")
for row in data_list[:5]:
    print(row)

First 5 rows of data (list of lists):
['#: table_doi: 10.17182/hepdata.155182.v2/t1']
['#: name: SI cross section']
['#: description: 90% CL WIMP SI cross sections', ' including sensitivities']
['#: data_file: SI_WS2022+WS2024.yaml']
['#: keyword reactions: chi N-> chi N']


In [3]:
data_sections = []
current_header = None
current_data = []

for row in data_list:
    if not row or row[0].strip().startswith('#:'): # Skip empty rows or metadata rows
        if current_header and current_data: # If we have a header and data, store the section
            data_sections.append({'header': current_header, 'data': current_data})
        current_header = None # Reset header and data for the next section
        current_data = []
        continue

    # If it's not metadata, check if it's a potential header
    # A simple heuristic: if the previous row was metadata or we are at the start
    # and this row contains strings that could be column names.
    if not current_header: # This is the first non-metadata row for a new section
        # Let's assume the first non-metadata row after a metadata block is the header
        current_header = [item.strip() for item in row]
    else:
        # Otherwise, it's data for the current section
        current_data.append([item.strip() for item in row])

# Add the last section if any data was collected
if current_header and current_data:
    data_sections.append({'header': current_header, 'data': current_data})

print("Identified Data Sections Headers:")
for i, section in enumerate(data_sections):
    print(f"Section {i+1} Header: {section['header']}")

# You can also inspect the raw sections structure if needed:
# print('\nFull Data Sections Structure:')
# import json
# print(json.dumps(data_sections, indent=2))


Identified Data Sections Headers:
Section 1 Header: ['mass', 'limit']
Section 2 Header: ['mass', 'limit_unconstr']
Section 3 Header: ['mass', '-2sigma']
Section 4 Header: ['mass', '-1sigma']
Section 5 Header: ['mass', 'median']
Section 6 Header: ['mass', '1sigma']
Section 7 Header: ['mass', '2sigma']
Section 8 Header: ['mass', 'median_3sigma_disco']


In [4]:
all_section_data = []

for section in data_sections:
    # The 'data' key in each section already contains a list of lists for that section
    all_section_data.append(section['data'])

print(f"Number of data sections extracted: {len(all_section_data)}")
print("First 2 rows of data from the first section:")
print(all_section_data[0][:2])

print("\nFirst 2 rows of data from the second section:")
print(all_section_data[1][:2])


Number of data sections extracted: 8
First 2 rows of data from the first section:
[['9.0', '9.797484060822151e-47'], ['11.0', '2.6404553011944104e-47']]

First 2 rows of data from the second section:
[['9.0', '6.360172460365096e-47'], ['11.0', '1.8701390104493043e-47']]


In [8]:
import json

structured_output_filename = 'structured_output.json'

with open(structured_output_filename, 'w') as f:
    json.dump(all_section_data, f, indent=4)

print(f"The structured data (list of lists of lists) has been written to '{structured_output_filename}'")

The structured data (list of lists of lists) has been written to 'structured_output.json'
